# L13a: REST, JSON, and Schemas

A network client is a program at a system boundary. Its result is only as reliable as the transport, payload, and application contracts it checks.

> **Learning objectives**
>
> - Read an HTTP request/response as a computational interface.
> - Follow a documented link instead of reconstructing an endpoint.
> - Distinguish transport, JSON, and application-schema failures.
> - Test the workflow without requiring a live service.


## Setup

Run the local setup cell first. It activates the pinned course environment, loads every package used by this meeting, and includes the `Week13Client` module from [`../src/Week13Client.jl`](../src/Week13Client.jl), which provides the functions called below. The recorded response fixtures used below are committed under [`../data`](../data).

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and includes the meeting's local source. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


## The linked-request workflow

The National Weather Service points endpoint does not return the forecast itself. It returns metadata containing a documented `properties.forecastHourly` link. The client follows that link and validates only the fields it promises downstream.

`engineering question -> points request -> linked forecast request -> validated records`


In [2]:
data_dir = normpath(joinpath(@__DIR__, "..", "data"))
points_payload = JSON.parsefile(joinpath(data_dir, "nws-points-ithaca.fixture.json"))
forecast_payload = JSON.parsefile(joinpath(data_dir, "nws-forecast-hourly-ithaca.fixture.json"))

points_url = build_points_url(42.443961, -76.501881)
forecast_url = parse_points_response(points_payload)
(points_url = points_url, linked_forecast_url = forecast_url)


(points_url = "https://api.weather.gov/points/42.443961,-76.501881", linked_forecast_url = "https://api.weather.gov/gridpoints/BGM/52,99/forecast/hourly")

In [3]:
periods = parse_forecast_response(forecast_payload)
DataFrame(periods)


Row,windDirection,shortForecast,startTime,isDaytime,windSpeed,temperatureUnit,endTime,temperature
,String,String,String,Bool,String,String,String,Int64
1,NW,Mostly Sunny,2026-08-10T09:00:00-04:00,true,5 mph,F,2026-08-10T10:00:00-04:00,72
2,NW,Partly Cloudy,2026-08-10T10:00:00-04:00,true,6 mph,F,2026-08-10T11:00:00-04:00,74


## Three layers of failure

| Layer | Example | Client response |
|---|---|---|
| Transport | HTTP 503 | Report the status and request URL |
| Serialization | malformed JSON | Report that parsing failed |
| Application schema | missing `forecastHourly` | Name the missing required field |

A successful HTTP status does not imply valid JSON, and valid JSON does not imply the fields our program requires.


In [4]:
required_fields = collect(Week13Client.REQUIRED_PERIOD_FIELDS)
DataFrame(field = required_fields, required_by_client = fill(true, length(required_fields)))


Row,field,required_by_client
,String,Bool
1,startTime,true
2,endTime,true
3,isDaytime,true
4,temperature,true
5,temperatureUnit,true
6,windSpeed,true
7,windDirection,true
8,shortForecast,true


## Deterministic versus live tests

Committed fixtures make the normal class path reproducible and let us force failure cases. A live call remains useful as an optional integration test, but service availability, changing observations, or a network policy must not block the lesson.

The client also requires HTTPS and a configurable identifying `User-Agent`. It uses no secret or API key.


## Summary

A defensive client does more than download bytes: it makes assumptions explicit, validates the boundary it owns, and produces errors that distinguish transport from data-contract failures.
